<a href="https://colab.research.google.com/github/kahli-creator/Kahli.f/blob/main/Kahli_load_hcp_WM_TASK_KAHLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Human Connectome Project (HCP) Dataset loader

The HCP dataset comprises resting-state and task-based fMRI from a large sample of human subjects. The NMA-curated dataset includes time series data that has been preprocessed and spatially-downsampled by aggregating within 360 regions of interest.

In [ ]:
# @title Install dependencies
!pip install nilearn --quiet

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Necessary for visualization
from nilearn import plotting, datasets

In [ ]:
#@title Figure settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/course-content/master/nma.mplstyle")

# Basic parameters

In [ ]:
# The download cells will store the data in nested directories starting here:
HCP_DIR = "./DATA"
if not os.path.isdir(HCP_DIR):
  os.mkdir(HCP_DIR)

# The data shared for NMA projects is a subset of the full HCP dataset
N_SUBJECTS = 339

# The data have already been aggregated into ROIs from the Glasesr parcellation
N_PARCELS = 360

# The acquisition parameters for all tasks were identical
TR = 0.72  # Time resolution, in sec

# The parcels are matched across hemispheres with the same order
HEMIS = ["Right", "Left"]

# Each experiment was repeated multiple times in each subject
N_RUNS_REST = 4
N_RUNS_TASK = 2

# Time series data are organized by experiment, with each experiment
# having an LR and RL (phase-encode direction) acquisition
BOLD_NAMES = [
  "rfMRI_REST1_LR", "rfMRI_REST1_RL",
  "rfMRI_REST2_LR", "rfMRI_REST2_RL",
  "tfMRI_MOTOR_RL", "tfMRI_MOTOR_LR",
  "tfMRI_WM_RL", "tfMRI_WM_LR",
  "tfMRI_EMOTION_RL", "tfMRI_EMOTION_LR",
  "tfMRI_GAMBLING_RL", "tfMRI_GAMBLING_LR",
  "tfMRI_LANGUAGE_RL", "tfMRI_LANGUAGE_LR",
  "tfMRI_RELATIONAL_RL", "tfMRI_RELATIONAL_LR",
  "tfMRI_SOCIAL_RL", "tfMRI_SOCIAL_LR"
]

# You may want to limit the subjects used during code development.
# This will use all subjects:
subjects = range(N_SUBJECTS)

# Downloading data

The rest and task data are shared in different files, but they will unpack into the same directory structure.

Each file is fairly large and will take some time to download. If you are focusing only on rest or task analyses, you may not want to download only that dataset.

We also separately provide some potentially useful behavioral covariate information.

In [ ]:
# @title Download the data

# @markdown Task data in `HCP_DIR/hcp_task`, rest in `HCP_DIR/hcp_rest`, covariate in `HCP_DIR/hcp`

import os, requests, tarfile

fnames = ["hcp_rest.tgz",
          "hcp_task.tgz",
          "hcp_covariates.tgz",
          "atlas.npz"]
urls = ["https://osf.io/bqp7m/download",
        "https://osf.io/s4h8j/download",
        "https://osf.io/x5p4g/download",
        "https://osf.io/j5kuc/download"]

for fname, url in zip(fnames, urls):
  if not os.path.isfile(fname):
    try:
      r = requests.get(url)
    except requests.ConnectionError:
      print("!!! Failed to download data !!!")
    else:
      if r.status_code != requests.codes.ok:
        print("!!! Failed to download data !!!")
      else:
        print(f"Downloading {fname}...")
        with open(fname, "wb") as fid:
          fid.write(r.content)
        print(f"Download {fname} completed!")

In [ ]:
# @title Extract the data in `HCP_DIR`
fnames = ["hcp_covariates", "hcp_rest", "hcp_task"]

for fname in fnames:
  # open file
  path_name = os.path.join(HCP_DIR, fname)
  if not os.path.exists(path_name):
    print(f"Extracting {fname}.tgz...")
    with tarfile.open(f"{fname}.tgz") as fzip:
      fzip.extractall(HCP_DIR)
  else:
    print(f"File {fname}.tgz has already been extracted.")

Extracting hcp_covariates.tgz...


/tmp/ipykernel_2100/1936258888.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  fzip.extractall(HCP_DIR)


Extracting hcp_rest.tgz...
Extracting hcp_task.tgz...


### Identify Common Subjects Across Rest and Task Data

To ensure we are working with subjects that have both resting-state and task-based fMRI data, we will find the intersection of subject IDs present in both `hcp_rest` and `hcp_task` directories.

In [ ]:
import glob

# Get subject directories for rest and task data
rest_subject_dirs = sorted([os.path.basename(d) for d in glob.glob(os.path.join(HCP_DIR, 'hcp_rest', 'subjects', '*'))])
task_subject_dirs = sorted([os.path.basename(d) for d in glob.glob(os.path.join(HCP_DIR, 'hcp_task', 'subjects', '*'))])

# Find subjects common to both datasets
common_subjects = sorted(list(set(rest_subject_dirs).intersection(set(task_subject_dirs))))

print(f"Found {len(common_subjects)} common subjects between hcp_rest and hcp_task.")
print(f"First 10 common subjects: {common_subjects[:10]}")

Found 339 common subjects between hcp_rest and hcp_task.
First 10 common subjects: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106']


### Locate Behavioral Data
We are checking the path `./DATA/hcp/behavior/` for the summary behavioral files and matching them with subjects.

### Fresh Identification of 100 Common Subjects
We will now cross-reference the subject IDs found in the behavior, rest, and task directories to ensure we only work with subjects present in all three.

In [ ]:
import os
import glob

# Define directory paths
behavior_subjects_path = os.path.join(HCP_DIR, 'hcp', 'subjects')
rest_subjects_path = os.path.join(HCP_DIR, 'hcp_rest', 'subjects')
task_subjects_path = os.path.join(HCP_DIR, 'hcp_task', 'subjects')

# Helper to get sub-folders (subject IDs), filtering out hidden files
def get_subject_ids(path):
    return set([os.path.basename(d) for d in glob.glob(os.path.join(path, '*')) if not os.path.basename(d).startswith('.')])

behavior_ids = get_subject_ids(behavior_subjects_path)
rest_ids = get_subject_ids(rest_subjects_path)
task_ids = get_subject_ids(task_subjects_path)

# Intersection of all three
common_100 = sorted(list(behavior_ids.intersection(rest_ids).intersection(task_ids)))

print(f"Subjects in Behavior folder: {len(behavior_ids)}")
print(f"Subjects in Rest folder: {len(rest_ids)}")
print(f"Subjects in Task folder: {len(task_ids)}")
print(f"\nNumber of common subjects found: {len(common_100)}")

# Limit to the first 100 if more are found, or as requested
final_subjects = common_100[:100]
print(f"Final list of 100 subjects selected: {final_subjects}")

Subjects in Behavior folder: 339
Subjects in Rest folder: 339
Subjects in Task folder: 339

Number of common subjects found: 339
Final list of 100 subjects selected: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '18', '180', '181', '182', '183', '184', '185', '186', '187', '188']


## Loading region information

Downloading either dataset will create the `regions.npy` file, which contains the region name and network assignment for each parcel.

Detailed information about the name used for each region is provided [in the Supplement](https://static-content.springer.com/esm/art%3A10.1038%2Fnature18933/MediaObjects/41586_2016_BFnature18933_MOESM330_ESM.pdf) to [Glasser et al. 2016](https://www.nature.com/articles/nature18933).

Information about the network parcellation is provided in [Ji et al, 2019](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6289683/).

### Loading Event-Related Data for 'WM' Task

We will use the `load_evs` function to retrieve the onset, duration, and amplitude for each trial of a specific condition within the 'WM' task. The `HCP_DIR` is set to `./DATA`.

In [ ]:
dir = os.path.join(HCP_DIR, "hcp_task")  # choose the data directory
regions = np.load(os.path.join(dir, "regions.npy")).T
region_info = dict(name=regions[0].tolist(),
                   network=regions[1],
                   myelin=regions[2].astype(float)
                   )

We also provide the [parcellation on the fsaverage5 surface](https://figshare.com/articles/HCP-MMP1_0_projected_on_fsaverage/3498446) and approximate MNI coordinates of each region, which can be useful for visualization:

In [ ]:
with np.load(f"atlas.npz") as dobj:
  atlas = dict(**dobj)

# Helper functions


## Data loading

In [ ]:
def get_image_ids(name):
  """Get the 1-based image indices for runs in a given experiment.

    Args:
      name (str) : Name of experiment ("rest" or name of task) to load
    Returns:
      run_ids (list of int) : Numeric ID for experiment image files

  """
  run_ids = [
             i for i, code in enumerate(BOLD_NAMES, 1) if name.upper() in code
             ]
  if not run_ids:
    raise ValueError(f"Found no data for '{name}'")
  return run_ids


def load_timeseries(subject, name, dir,
                    runs=None, concat=True, remove_mean= True):
  """Load timeseries data for a single subject.

  Args:
    subject (int): 0-based subject ID to load
    name (str) : Name of experiment ("rest" or name of task) to load
    dir (str) : data directory
    run (None or int or list of ints): 0-based run(s) of the task to load,
      or None to load all runs.
    concat (bool) : If True, concatenate multiple runs in time
    remove_mean (bool) : If True, subtract the parcel-wise mean

  Returns
    ts (n_parcel x n_tp array): Array of BOLD data values

  """
  # Get the list relative 0-based index of runs to use
  if runs is None:
    runs = range(N_RUNS_REST) if name == "rest" else range(N_RUNS_TASK)
  elif isinstance(runs, int):
    runs = [runs]

  # Get the first (1-based) run id for this experiment
  offset = get_image_ids(name)[0]

  # Load each run's data
  bold_data = [
               load_single_timeseries(subject,
                                      offset + run,
                                      dir,
                                      remove_mean) for run in runs
               ]

  # Optionally concatenate in time
  if concat:
    bold_data = np.concatenate(bold_data, axis=-1)

  return bold_data


def load_single_timeseries(subject, bold_run, dir, remove_mean=True):
  """Load timeseries data for a single subject and single run.

  Args:
    subject (int): 0-based subject ID to load
    bold_run (int): 1-based run index, across all tasks
    dir (str) : data directory
    remove_mean (bool): If True, subtract the parcel-wise mean

  Returns
    ts (n_parcel x n_timepoint array): Array of BOLD data values

  """
  bold_path = os.path.join(dir, "subjects", str(subject), "timeseries")
  bold_file = f"bold{bold_run}_Atlas_MSMAll_Glasser360Cortical.npy"
  ts = np.load(os.path.join(bold_path, bold_file))
  if remove_mean:
    ts -= ts.mean(axis=1, keepdims=True)
  return ts


def load_evs(subject, name, condition, dir):
  """Load EV (explanatory variable) data for one task condition.

  Args:
    subject (int): 0-based subject ID to load
    name (str) : Name of task
    condition (str) : Name of condition
    dir (str) : data directory

  Returns
    evs (list of dicts): A dictionary with the onset, duration, and amplitude
      of the condition for each run.

  """
  evs = []
  for id in get_image_ids(name):
    task_key = BOLD_NAMES[id - 1]
    ev_file = os.path.join(dir, "subjects", str(subject), "EVs",
                           task_key, f"{condition}.txt")
    ev_array = np.loadtxt(ev_file, ndmin=2, unpack=True)
    ev = dict(zip(["onset", "duration", "amplitude"], ev_array))
    evs.append(ev)
  return evs

In [ ]:
subject = final_subjects[0]

evs1 = load_evs(
    subject,
    "wm",
    "2bk_body",
    dir=os.path.join(HCP_DIR, "hcp_task")
)

print(evs1)

[{'onset': array([7.997]), 'duration': array([27.5]), 'amplitude': array([1.])}, {'onset': array([150.593]), 'duration': array([27.5]), 'amplitude': array([1.])}]


In [ ]:
print(type(evs1))

<class 'list'>


In [ ]:
print(evs1[0])

{'onset': array([7.997]), 'duration': array([27.5]), 'amplitude': array([1.])}


In [ ]:
print("Onsets:", evs1[0]["onset"])
print("Durations:", evs1[0]["duration"])
print("Amplitudes:", evs1[0]["amplitude"])

Onsets: [7.997]
Durations: [27.5]
Amplitudes: [1.]


In [ ]:
print("Run 2 onsets:", evs1[1]["onset"])

Run 2 onsets: [150.593]


In [ ]:
print(evs1[0]["onset"])
print(evs1[0]["duration"])
print(evs1[0]["amplitude"])

[7.997]
[27.5]
[1.]


In [ ]:
print(type(evs1))

<class 'list'>


## Task-based analysis

In [ ]:
def condition_frames(run_evs, skip=0):
  """Identify timepoints corresponding to a given condition in each run.

  Args:
    run_evs (list of dicts) : Onset and duration of the event, per run
    skip (int) : Ignore this many frames at the start of each trial, to account
      for hemodynamic lag

  Returns:
    frames_list (list of 1D arrays): Flat arrays of frame indices, per run

  """
  frames_list = []
  for ev in run_evs:

    # Determine when trial starts, rounded down
    start = np.floor(ev["onset"] / TR).astype(int)

    # Use trial duration to determine how many frames to include for trial
    duration = np.ceil(ev["duration"] / TR).astype(int)

    # Take the range of frames that correspond to this specific trial
    frames = [s + np.arange(skip, d) for s, d in zip(start, duration)]

    frames_list.append(np.concatenate(frames))

  return frames_list


def selective_average(timeseries_data, ev, skip=0):
  """Take the temporal mean across frames for a given condition.

  Args:
    timeseries_data (array or list of arrays): n_parcel x n_tp arrays
    ev (dict or list of dicts): Condition timing information
    skip (int) : Ignore this many frames at the start of each trial, to account
      for hemodynamic lag

  Returns:
    avg_data (1D array): Data averagted across selected image frames based
    on condition timing

  """
  # Ensure that we have lists of the same length
  if not isinstance(timeseries_data, list):
    timeseries_data = [timeseries_data]
  if not isinstance(ev, list):
    ev = [ev]
  if len(timeseries_data) != len(ev):
    raise ValueError("Length of `timeseries_data` and `ev` must match.")

  # Identify the indices of relevant frames
  frames = condition_frames(ev, skip)

  # Select the frames from each image
  selected_data = []
  for run_data, run_frames in zip(timeseries_data, frames):
    run_frames = run_frames[run_frames < run_data.shape[1]]
    selected_data.append(run_data[:, run_frames])

  # Take the average in each parcel
  avg_data = np.concatenate(selected_data, axis=-1).mean(axis=-1)

  return avg_data

# Resting-state analyses

Load a single run of resting-state data:

In [ ]:
help(load_timeseries)

Help on function load_timeseries in module __main__:

load_timeseries(subject, name, dir, runs=None, concat=True, remove_mean=True)
    Load timeseries data for a single subject.

    Args:
      subject (int): 0-based subject ID to load
      name (str) : Name of experiment ("rest" or name of task) to load
      dir (str) : data directory
      run (None or int or list of ints): 0-based run(s) of the task to load,
        or None to load all runs.
      concat (bool) : If True, concatenate multiple runs in time
      remove_mean (bool) : If True, subtract the parcel-wise mean

    Returns
      ts (n_parcel x n_tp array): Array of BOLD data values



In [ ]:
timeseries = load_timeseries(subject=0,
                             name="rest",
                             dir=os.path.join(HCP_DIR, "hcp_rest"),
                             runs=1)
print(timeseries.shape)  # n_parcel x n_timepoint

(360, 1200)


In [ ]:
print(N_SUBJECTS)

339


In [ ]:
print(timeseries)

[[-137.14958333 -212.84958333 -266.64958333 ...    6.15041667
    26.85041667   60.05041667]
 [-112.33685833 -183.79685833 -209.09685833 ...  -62.52685833
   -10.31685833  -48.12685833]
 [-130.10033333 -159.80033333 -216.50033333 ...  -58.90033333
   -86.70033333  -87.80033333]
 ...
 [  91.80741667   72.90741667  -34.99258333 ... -143.69258333
   -31.99258333  -60.49258333]
 [-113.99291667 -220.89291667 -249.89291667 ...   70.90708333
    35.90708333   96.80708333]
 [ -51.65058333 -122.35058333  -41.35058333 ...  -20.75058333
    55.34941667    6.24941667]]


Load a concatenated resting-state timeseries (using all runs' data) for each subject:

In [ ]:
timeseries_rest = []
for subject in final_subjects:
  ts_concat = load_timeseries(subject, name="rest",
                              dir=os.path.join(HCP_DIR, "hcp_rest"))
  timeseries_rest.append(ts_concat)

In [ ]:
final_subjects = [int(s) for s in final_subjects]

In [ ]:
print(final_subjects)

[0, 1, 10, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 11, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 12, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 13, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 14, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 15, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 16, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 17, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18, 180, 181, 182, 183, 184, 185, 186, 187, 188]


In [ ]:
wm_behavior = np.genfromtxt(os.path.join(HCP_DIR, "hcp", "behavior", "wm.csv"),
                            delimiter=",",
                            names=True,
                            dtype=None,
                            encoding="utf")

# Filter wm_behavior for the 100 final_subjects
# Convert final_subjects to a set for efficient lookup
final_subjects_set = set(map(int, final_subjects))

# 'Subject' is the column name for subject IDs in wm_behavior

filtered_wm_behavior = [row for row in wm_behavior if row['Subject'] in final_subjects_set]

# Convert the filtered list back to a structured numpy array
# This assumes all rows have the same dtype. If not, further handling might be needed.
filtered_wm_behavior = np.array(filtered_wm_behavior, dtype=wm_behavior.dtype)

print(filtered_wm_behavior[:5])
print(filtered_wm_behavior.dtype.names)

**Resting state connectivity comparaison (high VS low)**

In [ ]:
#from google.colab import files
#uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
!pip install openpyxl

In [ ]:
"""import pandas as pd

behavior_df = pd.read_excel("hcp_behavioral_data.xlsx")
behavior_df.head()"""

In [ ]:
# every downstream calculation and plot is based on clean data.
df = behavior_df[behavior_df["Subject"] != 143].copy()
print(f"After removing Subject 143: {df.Subject.nunique()} subjects, {len(df)} rows")

In [ ]:
behavior_df.columns

In [ ]:
behavior_df["IES"] = behavior_df["MEDIAN_RT"] / behavior_df["ACC"]

In [ ]:
print(behavior_df.columns)

In [ ]:
subject_scores = (
    behavior_df
    .groupby("Subject")["IES"]
    .mean()
    .reset_index()
)

subject_scores.head()

In [ ]:
behavior_df["load"] = behavior_df["ConditionName"].str[:3]

In [ ]:
subject_ies = (
    behavior_df
    .groupby(["Subject", "load"])["IES"]
    .mean()
    .unstack()
)

subject_ies.head()

In [ ]:
subject_ies["IES_cost"] = subject_ies["2BK"] - subject_ies["0BK"]

In [ ]:
clean_df = subject_ies.dropna().reset_index()

In [ ]:
# ---- Split into HIGH and LOW performance using 25th/75th percentile ----
q25 = clean_df.IES_cost.quantile(0.25)
q75 = clean_df.IES_cost.quantile(0.75)

clean_df["performance_group"] = "middle"
clean_df.loc[clean_df.IES_cost <= q25, "performance_group"] = "high_performance"
clean_df.loc[clean_df.IES_cost >= q75, "performance_group"] = "low_performance"

n_high = sum(clean_df.performance_group == "high_performance")
n_low = sum(clean_df.performance_group == "low_performance")
n_mid = sum(clean_df.performance_group == "middle")

print(f"\n25th percentile cutoff (high performers below this): {q25:.1f}")
print(f"75th percentile cutoff (low performers above this):  {q75:.1f}")
print(f"\nHigh performance: n={n_high}")
print(f"Low performance:  n={n_low}")
print(f"Middle (excluded): n={n_mid}")


In [ ]:
import numpy as np

print(type(timeseries_rest))
print(np.shape(timeseries_rest))

In [ ]:
import numpy as np

n_subjects = len(timeseries_rest)
n_regions = timeseries_rest[0].shape[0]

fc = np.zeros((n_subjects, n_regions, n_regions))

for i, ts in enumerate(timeseries_rest):
    fc[i] = np.corrcoef(ts)

print(fc.shape)

In [ ]:
high_subjects = clean_df.loc[
    clean_df["performance_group"] == "high_performance",
    "Subject"
].tolist()

low_subjects = clean_df.loc[
    clean_df["performance_group"] == "low_performance",
    "Subject"
].tolist()

print("High performers:", len(high_subjects))
print("Low performers:", len(low_subjects))
print(high_subjects[:10])
print(low_subjects[:10])

In [ ]:
subject_to_index = {
    subject: i
    for i, subject in enumerate(final_subjects)
}

In [ ]:
high_idx = [
    subject_to_index[s]
    for s in high_subjects
    if s in subject_to_index
]

low_idx = [
    subject_to_index[s]
    for s in low_subjects
    if s in subject_to_index
]

In [ ]:
print("High:", len(high_idx))
print("Low :", len(low_idx))

print(high_idx[:10])
print(low_idx[:10])

In [ ]:
fc_high = fc[high_idx]
fc_low = fc[low_idx]

In [ ]:
mean_fc_high = fc_high.mean(axis=0)
mean_fc_low = fc_low.mean(axis=0)

fc_difference = mean_fc_high - mean_fc_low

In [ ]:
print(mean_fc_high.shape)
print(mean_fc_low.shape)
print(fc_difference.shape)

In [ ]:
fc_difference = mean_fc_high - mean_fc_low

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.imshow(fc_difference, cmap="bwr", vmin=-0.3, vmax=0.3)
plt.colorbar(label="Connectivity Difference (High − Low)")
plt.title("Group Resting-State Connectivity Difference Matrix")
plt.xlabel("Brain Regions")
plt.ylabel("Brain Regions")
plt.show()

**option 2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

high_mean = np.mean(fc_high[:, np.triu_indices(360, k=1)[0],
                              np.triu_indices(360, k=1)[1]], axis=1)

low_mean = np.mean(fc_low[:, np.triu_indices(360, k=1)[0],
                            np.triu_indices(360, k=1)[1]], axis=1)

plt.figure(figsize=(5,6))
plt.boxplot([high_mean, low_mean],
            labels=["High", "Low"])

plt.ylabel("Mean Resting-State FC")
plt.title("Overall Functional Connectivity")
plt.show()

## Run a simple correlation-based "functional connectivity" analysis

Generate a correlation matrix (showing "functional connectivity" or FC) for each subject and plot the group average:

In [ ]:
fc = np.zeros((N_SUBJECTS, N_PARCELS, N_PARCELS))
for sub, ts in enumerate(timeseries_rest):
  fc[sub] = np.corrcoef(ts)

group_fc = fc.mean(axis=0)

plt.figure()
plt.imshow(group_fc, interpolation="none", cmap="bwr", vmin=-1, vmax=1)
plt.colorbar()
plt.show()

Plot the profile of FC values between a particular "seed" parcel and every parcel in the dataset, separated by hemisphere:

In [ ]:
seed_roi = "R_FEF"  # name of seed parcel
ind = region_info["name"].index(seed_roi)

hemi_fc = np.split(group_fc, 2)

# Plot the FC profile across the right and left hemisphere target regions
plt.figure()
for i, hemi_fc in enumerate(hemi_fc):
  plt.plot(hemi_fc[:, ind], label=f"{HEMIS[i]} hemisphere")
plt.title(f"FC for region {seed_roi}")
plt.xlabel("Target region")
plt.ylabel("Correlation (FC)")
plt.legend()
plt.show()

Threshold the correlation matrix to produce a connectome, and plot it:

In [ ]:
plotting.view_connectome(group_fc, atlas["coords"], edge_threshold="99%")

**resting state connectivity difference**
**High VS Low**

# Working memory project

The rest of this notebook is a single, focused study of the **working-memory
(WM) N-back task**. Every analysis below is organized around one contrast,
`2-back - 0-back`, which isolates the cost of *holding and updating*
information while stimulus category, motor response, and timing are held
essentially constant.

**Motivation.** Working memory is capacity-limited, and that limit shapes
reasoning, learning, and language and tracks fluid intelligence. The load
manipulation changes only the demand to maintain and update, so whatever the
cortex does differently between 2-back and 0-back is close to the mechanism of
the capacity itself. WM deficits are transdiagnostic (schizophrenia, ADHD,
aging), so a brain measure that tracks load *and* predicts an individual's
performance is a candidate biomarker. The HCP is well suited because it
combines a within-subject load manipulation, trial-level accuracy, and a large
sample.

We investigate five aspects of the same idea:
1. the group **load map** (`2bk - 0bk`) and its network profile,
2. whether the load cost is **content-general or content-specific** (per category),
3. where **success and failure** diverge under high load (`2bk_err - 2bk_cor`),
4. how **network integration** reconfigures under load (task FC vs rest FC),
5. whether load-response features **predict WM performance** (`wm.csv`).

## GLM helpers

`build_events` turns the EV timing files into a per-run events table;
`averaged_contrast` builds a mean(pos) - mean(neg) contrast vector padded with
zeros for the drift/constant columns; `fit_subject_contrasts` fits the design
**once per run** and reads out several contrasts from that single fit.

In [ ]:
import os
import pandas as pd
from nilearn.glm.first_level import make_first_level_design_matrix, run_glm
from nilearn.glm.contrasts import compute_contrast


def list_ev_conditions(subject, task, dir):
    """Return the EV condition names actually present on disk for a task."""
    task_key = BOLD_NAMES[get_image_ids(task)[0] - 1]
    ev_dir = os.path.join(dir, "subjects", str(subject), "EVs", task_key)
    return sorted(f[:-4] for f in os.listdir(ev_dir) if f.endswith(".txt"))


def build_events(subject, task, conditions, dir):
    """One events DataFrame (onset / duration / trial_type) per run.

    Robust to empty EV files (e.g. a run with no 2bk error trials), which
    load_evs returns as a dict missing the 'duration'/'onset' keys.
    """
    ev_by_cond = {c: load_evs(subject, task, c, dir) for c in conditions}
    n_runs = len(next(iter(ev_by_cond.values())))
    events_per_run = []
    for r in range(n_runs):
        rows = []
        for c in conditions:
            ev = ev_by_cond[c][r]
            onsets = np.atleast_1d(ev.get("onset", []))
            durations = np.atleast_1d(ev.get("duration", []))
            n = min(len(onsets), len(durations))     # skip if either is empty
            for onset, dur in zip(onsets[:n], durations[:n]):
                rows.append({"trial_type": c, "onset": onset, "duration": dur})
        events_per_run.append(pd.DataFrame(rows))
    return events_per_run


def averaged_contrast(design_columns, pos, neg):
    """mean(pos) - mean(neg) contrast vector, or None if a side is absent."""
    cols = list(design_columns)
    pos = [c for c in pos if c in cols]
    neg = [c for c in neg if c in cols]
    if not pos or not neg:
        return None
    con = np.zeros(len(cols))
    for c in pos:
        con[cols.index(c)] = 1.0 / len(pos)
    for c in neg:
        con[cols.index(c)] = -1.0 / len(neg)
    return con


def fit_subject_contrasts(subject, task, model_conditions, contrasts, dir):
    """Fit the per-run GLM once; return {name: (effect, z)} averaged over runs.

    contrasts : dict name -> (pos_list, neg_list)
    """
    ts_runs = load_timeseries(subject, task, dir, concat=False)
    events_per_run = build_events(subject, task, model_conditions, dir)

    acc = {name: {"eff": [], "z": []} for name in contrasts}
    for ts, events in zip(ts_runs, events_per_run):
        frame_times = TR * np.arange(ts.shape[1])
        X = make_first_level_design_matrix(
            frame_times, events,
            hrf_model="spm", drift_model="cosine", high_pass=0.01,
        )
        labels, estimates = run_glm(ts.T, X.values, noise_model="ols")

        for name, (pos, neg) in contrasts.items():
            con_val = averaged_contrast(X.columns, pos, neg)
            if con_val is None:
                acc[name]["eff"].append(np.full(N_PARCELS, np.nan))
                acc[name]["z"].append(np.full(N_PARCELS, np.nan))
                continue
            con = compute_contrast(labels, estimates, con_val, stat_type="t")
            acc[name]["eff"].append(np.asarray(con.effect_size()).ravel())
            acc[name]["z"].append(np.asarray(con.z_score()).ravel())

    return {name: (np.nanmean(a["eff"], axis=0), np.nanmean(a["z"], axis=0))
            for name, a in acc.items()}

## Discover the real condition names

The WM task ships several *overlapping* EV sets (stimulus categories, accuracy
splits, `all_bk_*`). They describe the same trials grouped differently, so we
never put them in one design together. Inspect the printout and confirm the
category names before modeling.

In [ ]:
task = "wm"
task_dir = os.path.join(HCP_DIR, "hcp_task")

avail = list_ev_conditions(0, task, task_dir)
print("Available WM conditions:", avail)

# Stimulus-category regressors only (drop accuracy / no-response bookkeeping EVs)
cat = [c for c in avail
       if (c.startswith("0bk_") or c.startswith("2bk_"))
       and not c.endswith(("_cor", "_err", "_nlr", "_nir"))]

pos = [c for c in cat if c.startswith("2bk_")]   # high load
neg = [c for c in cat if c.startswith("0bk_")]   # low load

# Categories present as BOTH 0bk and 2bk -> per-category contrasts (aspect 2)
bases = sorted({c.split("_", 1)[1] for c in cat})
bases = [b for b in bases if f"2bk_{b}" in cat and f"0bk_{b}" in cat]

print("Modeling conditions:", cat)
print("Load contrast:", pos, "minus", neg)
print("Per-category bases:", bases)

## Aspects 1 & 2 - load map and content-specificity

A single GLM per run yields the overall load contrast and each per-category
load contrast at once. We keep every subject's map so we can build individual
features later (aspect 5).

In [ ]:
contrasts = {"load": (pos, neg)}
for b in bases:
    contrasts[f"load_{b}"] = ([f"2bk_{b}"], [f"0bk_{b}"])

# effect maps per contrast: subject x parcel
eff_by_con = {name: [] for name in contrasts}
z_by_con = {name: [] for name in contrasts}

for subject in subjects:
    res = fit_subject_contrasts(subject, task, cat, contrasts, task_dir)
    for name, (eff, z) in res.items():
        eff_by_con[name].append(eff)
        z_by_con[name].append(z)

# Stack to arrays and average to group level
load_effects = np.asarray(eff_by_con["load"])          # (n_subjects x 360)
group_load_contrast = np.nanmean(load_effects, axis=0)  # group load map
group_load_z = np.nanmean(z_by_con["load"], axis=0)

print("Per-subject load maps:", load_effects.shape)

In [ ]:
#start with differences , every region of interest need to take the variable. in this variable take teh time period


In [ ]:
import numpy as np

fpn_idx = np.where(region_info["network"] == "Frontopariet")[0]

fpn_names = np.array(region_info["name"])[fpn_idx]

print(f"Number of FPN parcels: {len(fpn_names)}")
print(fpn_names)

In [ ]:
fpn_idx = np.where(region_info["network"] == "Frontopariet")[0]

**BOLD Signal Analysis acroos all the FPN Network**

In [ ]:
subject_to_analyze = 0
task_to_analyze = "wm"

# Load unconcatenated time series once for the chosen subject and task
timeseries_runs = load_timeseries(subject_to_analyze, task_to_analyze,
                                  dir=os.path.join(HCP_DIR, "hcp_task"),
                                  concat=False)

condition1 = '2bk_body'
condition2 = '0bk_body'

# Load event information for both conditions once
evs1 = load_evs(subject_to_analyze, task_to_analyze, condition1, dir=os.path.join(HCP_DIR, "hcp_task"))
evs2 = load_evs(subject_to_analyze, task_to_analyze, condition2, dir=os.path.join(HCP_DIR, "hcp_task"))

print(f"Analyzing task: {task_to_analyze}")
print(f"Number of runs for this task: {len(timeseries_runs)}")
print(f"Comparing conditions: {condition1} vs {condition2}\n")

# Loop through every region
for region_index in fpn_idx:

    region_name = region_info["name"][region_index]

    print(f"--- Analyzing Region: {region_name} (index {region_index}) ---")

    # #### Raw Time Series for the Selected Region Across Runs
    plt.figure(figsize=(15, 6))
    for i, ts_run in enumerate(timeseries_runs):
        plt.plot(ts_run[region_index, :], label=f'Run {i+1}')

    plt.title(f'Time Series for {region_name} (Subject {subject_to_analyze}, Task: {task_to_analyze})')
    plt.xlabel('Time Points (TRs)')
    plt.ylabel('BOLD Signal Change')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # #### Differences in Average Activity for Specific Conditions
    avg_cond1 = selective_average(timeseries_runs, evs1)[region_index]
    avg_cond2 = selective_average(timeseries_runs, evs2)[region_index]

    print(f"Average activity in {region_name} during {condition1}: {avg_cond1:.2f}")
    print(f"Average activity in {region_name} during {condition2}: {avg_cond2:.2f}")
    print(f"Difference ({condition1} - {condition2}): {avg_cond1 - avg_cond2:.2f}")

    plt.figure(figsize=(8, 5))
    plt.bar([condition1, condition2], [avg_cond1, avg_cond2], color=['skyblue', 'lightcoral'])
    plt.title(f'Average BOLD Signal for {region_name} (Subject {subject_to_analyze})')
    plt.xlabel('Condition')
    plt.ylabel('Average BOLD Signal Change')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    print("\n") # Add a newline for better separation between regions

### Exploring Time Series for a Specific Region During Task Conditions

Let's pick a subject, a task, and a region to visualize the raw time series data and then observe differences across conditions within that region.

In [ ]:
subject_to_analyze = 0 # First subject
task_to_analyze = "wm"

# Load unconcatenated time series for the chosen subject and task
timeseries_runs = load_timeseries(subject_to_analyze, task_to_analyze,
                                  dir=os.path.join(HCP_DIR, "hcp_task"),
                                  concat=False)

# Select a region of interest (e.g., R_FEF from previous analyses)
seed_roi_name = "R_FEF"
seed_roi_index = region_info["name"].index(seed_roi_name)

print(f"Analyzing task: {task_to_analyze}")
print(f"Selected region: {seed_roi_name} (index {seed_roi_index})")
print(f"Number of runs for this task: {len(timeseries_runs)}")


#### Raw Time Series for the Selected Region Across Runs

Here, we visualize the activity of the selected region over time for each run of the working memory task.

In [ ]:
plt.figure(figsize=(15, 6))
for i, ts_run in enumerate(timeseries_runs):
    plt.plot(ts_run[seed_roi_index, :], label=f'Run {i+1}')

plt.title(f'Time Series for {seed_roi_name} (Subject {subject_to_analyze}, Task: {task_to_analyze})')
plt.xlabel('Time Points (TRs)')
plt.ylabel('BOLD Signal Change')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


Exploring Time Series for FPN ROI`s

In [ ]:
subject_to_analyze = 0
task_to_analyze = "wm"

timeseries_runs = load_timeseries(
    subject_to_analyze,
    task_to_analyze,
    dir=os.path.join(HCP_DIR, "hcp_task"),
    concat=False
)

In [ ]:
print(f"Number of runs: {len(timeseries_runs)}")

for i, run in enumerate(timeseries_runs):
    print(f"Run {i+1}: {run.shape}")

In [ ]:
fpn_bold_runs = []

for run in timeseries_runs:
    fpn_bold = run[:, fpn_idx]   # Select only the 50 FPN parcels
    fpn_bold_runs.append(fpn_bold)

In [ ]:
fpn_mean_runs = []

for fpn_bold in fpn_bold_runs:
    fpn_mean = np.mean(fpn_bold, axis=1)
    fpn_mean_runs.append(fpn_mean)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

for i, fpn_mean in enumerate(fpn_mean_runs):
    plt.plot(fpn_mean, label=f"Run {i+1}")

plt.xlabel("Time (TR)")
plt.ylabel("Mean BOLD")
plt.title("Mean Frontoparietal Network (FPN) BOLD Signal")
plt.legend()
plt.show()

#### Differences in Average Activity for Specific Conditions

Now, let's use the `selective_average` function to calculate the mean activity in our chosen region during two contrasting conditions (e.g., `2bk_body` vs `0bk_body`) within the working memory task. This directly addresses the 'differences' aspect within a 'time period' relevant to each condition.

In [ ]:
condition1 = '2bk_body'
condition2 = '0bk_body'

# Load event information for both conditions
evs1 = load_evs(subject_to_analyze, task_to_analyze, condition1, dir=os.path.join(HCP_DIR, "hcp_task"))
evs2 = load_evs(subject_to_analyze, task_to_analyze, condition2, dir=os.path.join(HCP_DIR, "hcp_task"))

# Calculate the average activity in the selected region during each condition
avg_cond1 = selective_average(timeseries_runs, evs1)[seed_roi_index]
avg_cond2 = selective_average(timeseries_runs, evs2)[seed_roi_index]

print(f"Average activity in {seed_roi_name} during {condition1}: {avg_cond1:.2f}")
print(f"Average activity in {seed_roi_name} during {condition2}: {avg_cond2:.2f}")
print(f"Difference ({condition1} - {condition2}): {avg_cond1 - avg_cond2:.2f}")

# We can also plot these average values
plt.figure(figsize=(8, 5))
plt.bar([condition1, condition2], [avg_cond1, avg_cond2], color=['skyblue', 'lightcoral'])
plt.title(f'Average BOLD Signal for {seed_roi_name} (Subject {subject_to_analyze})')
plt.xlabel('Condition')
plt.ylabel('Average BOLD Signal Change')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


Plot the group load map across regions, on the surface, and by network:

In [ ]:
# Region-wise, by hemisphere
hemi_contrasts = np.split(group_load_contrast, 2)
plt.figure()
for i, hc in enumerate(hemi_contrasts):
    plt.plot(hc, label=f"{HEMIS[i]} hemisphere")
plt.title("Working-memory load: 2bk - 0bk")
plt.xlabel("Region")
plt.ylabel("Effect size (beta)")
plt.legend()
plt.show()

In [ ]:
# Surface projection of the load z-map (left hemisphere)
fsaverage = datasets.fetch_surf_fsaverage()
surf_z = group_load_z[atlas["labels_L"]]
plotting.view_surf(fsaverage["infl_left"], surf_z, vmax=8)

In [ ]:
# Network profile of the load effect (expect frontoparietal / dorsal-attention up,
# default-mode down)
network_names = np.unique(region_info["network"])
hemi_networks = np.split(region_info["network"], 2)
hemi_contrasts = np.split(group_load_contrast, 2)

plt.figure()
for hemi, hnet, hc in zip(HEMIS, hemi_networks, hemi_contrasts):
    vals = [hc[hnet == net].mean() for net in network_names]
    plt.barh(network_names, vals, alpha=.5, label=f"{hemi} hemi")
plt.axvline(0)
plt.xlabel("Mean load effect (2bk - 0bk)")
plt.title("Load effect by functional network")
plt.legend()
plt.show()

### Content-general or content-specific?

Compare the per-category load maps. If the load cost is content-general the
category maps are highly correlated; category-specific extra recruitment shows
up as departures from the common pattern.

In [ ]:
if bases:
    group_cat = {b: np.nanmean(eff_by_con[f"load_{b}"], axis=0) for b in bases}
    # Correlation between per-category load maps
    M = np.vstack([group_cat[b] for b in bases])
    C = np.corrcoef(M)
    plt.figure()
    plt.imshow(C, vmin=0, vmax=1, cmap="viridis")
    plt.xticks(range(len(bases)), bases, rotation=45, ha="right")
    plt.yticks(range(len(bases)), bases)
    plt.colorbar(label="spatial correlation of load map")
    plt.title("Load cost across stimulus categories")
    plt.tight_layout()
    plt.show()
    print("Mean off-diagonal correlation:",
          C[np.triu_indices(len(bases), 1)].mean())

## Aspect 3 - success vs failure under high load

`2bk_cor` and `2bk_err` overlap the category EVs, so this needs its **own**
model. The `2bk_err - 2bk_cor` contrast marks where activity diverges when the
subject fails - a signature of the capacity being exceeded. Some runs have no
error trials; those contribute NaN and are dropped by the nan-aware averaging.

In [ ]:
err_conditions = [c for c in avail if c in ("2bk_cor", "2bk_err")]

group_err = None
if set(("2bk_cor", "2bk_err")).issubset(avail):
    err_eff = []
    for subject in subjects:
        res = fit_subject_contrasts(
            subject, task, err_conditions,
            {"err": (["2bk_err"], ["2bk_cor"])}, task_dir,
        )
        err_eff.append(res["err"][0])
    err_eff = np.asarray(err_eff)
    group_err = np.nanmean(err_eff, axis=0)

    hemi = np.split(group_err, 2)
    plt.figure(figsize = (12,5))
    for i, h in enumerate(hemi):
        plt.plot(h, label=f"{HEMIS[i]} hemisphere")
    plt.title("2bk error - correct")
    plt.xlabel("Region")
    plt.ylabel("Effect size (beta)")
    plt.legend()
    plt.show()
else:
    print("Accuracy-split EVs not available; skipping aspect 3.")

## Aspect 4 - network reconfiguration under load

Does frontoparietal integration increase during the demanding task relative to
rest? Compute functional connectivity on the concatenated WM runs and compare
mean within-frontoparietal FC against the resting-state matrix (`fc` from the
rest branch).

In [ ]:
# Identify the frontoparietal network label by substring match
fp = [n for n in network_names
      if ("front" in n.lower() and "pariet" in n.lower()) or "fronto" in n.lower()]
fp_net = fp[0] if fp else None
print("Frontoparietal network label:", fp_net)

if fp_net is not None:
    fp_idx = np.where(region_info["network"] == fp_net)[0]

    def within_fp(mat):
        sub = mat[np.ix_(fp_idx, fp_idx)]
        iu = np.triu_indices(len(fp_idx), 1)
        return sub[iu].mean()

    rest_fp = np.array([within_fp(fc[s]) for s in range(N_SUBJECTS)])

    task_fp = []
    for subject in subjects:
        ts = load_timeseries(subject, "wm", task_dir)   # concatenated WM runs
        task_fp.append(within_fp(np.corrcoef(ts)))
    task_fp = np.array(task_fp)

    plt.figure()
    plt.hist(rest_fp, bins=30, alpha=.5, label="rest")
    plt.hist(task_fp, bins=30, alpha=.5, label="WM task")
    plt.xlabel("Mean within-frontoparietal FC")
    plt.ylabel("Subjects")
    plt.title("Frontoparietal integration: rest vs WM")
    plt.legend()
    plt.show()

    from scipy import stats
    t, p = stats.ttest_rel(task_fp, rest_fp)
    print(f"Paired t = {t:.2f}, p = {p:.2e} (task - rest)")

## Aspect 5 - predicting WM performance

The payoff. Use each subject's load-contrast features (mean 2bk-0bk effect
within each functional network) to predict their **actual** WM performance from
`wm.csv` with a statsmodels GLM. This uses the real behavioral file, not the
synthetic pseudo-demographics.

In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.api as sm

In [ ]:
import statsmodels.api as sm

wm_behavior = np.genfromtxt(os.path.join(HCP_DIR, "hcp", "behavior", "wm.csv"),
                            delimiter=",", names=True, dtype=None, encoding="utf")
print("wm.csv columns:", wm_behavior.dtype.names)


def pick_target(rec):
    """Choose a numeric behavioral column: prefer accuracy, else first numeric."""
    numeric = [n for n in rec.dtype.names if rec[n].dtype.kind in "fiu"
               and n.lower() not in ("subject", "id", "subject_id")]
    for key in ("acc", "pc", "perf", "rate", "correct"):
        for n in numeric:
            if key in n.lower():
                return n
    return numeric[0]


target_col = pick_target(wm_behavior)
print("Predicting:", target_col)
y_full = wm_behavior[target_col].astype(float)

In [ ]:
# Per-subject features: mean load effect within each network
feat = np.zeros((load_effects.shape[0], len(network_names)))
for j, net in enumerate(network_names):
    idx = np.where(region_info["network"] == net)[0]
    feat[:, j] = np.nanmean(load_effects[:, idx], axis=1)

# Align features and behavior, drop any rows with NaNs
m = min(len(feat), len(y_full))
Xf, y = feat[:m], y_full[:m]
ok = np.isfinite(y) & np.isfinite(Xf).all(axis=1)
Xf, y = Xf[ok], y[ok]

Xd = sm.add_constant(Xf)
# Gaussian identity link for a continuous score; swap for Poisson on counts
model = sm.GLM(y, Xd, family=sm.families.Gaussian())
res = model.fit()
print(res.summary())

In [ ]:
y_hat = res.predict(Xd)
plt.figure()
plt.scatter(y, y_hat, s=12, alpha=.5)
lims = [min(y.min(), y_hat.min()), max(y.max(), y_hat.max())]
plt.plot(lims, lims, "k--", lw=.7)
plt.xlabel(f"Observed ({target_col})")
plt.ylabel("Predicted")
plt.title("Load-response features -> WM performance")
plt.show()

r = np.corrcoef(y, y_hat)[0, 1]
print(f"In-sample predicted-observed r = {r:.3f}")

**Reading the result.** A positive frontoparietal coefficient means larger
load recruitment goes with better performance; a negative one is the *neural
efficiency* pattern (good performers recruit less). Note this is an in-sample
fit for interpretation - for a genuine predictive claim, wrap the GLM in
cross-validation across subjects.

### Is the prediction real? Cross-validation + permutation

The in-sample fit above is optimistic: with ~12 correlated network features,
OLS explains some variance by chance, so its predicted-observed `r` overstates
the effect. The honest number is the **out-of-sample** correlation under
cross-validation, judged against a **permutation null** (shuffle the behavioral
labels, redo the whole CV, and see how often chance does this well).

We use ridge regression inside a pipeline so the collinear features are
regularized and standardized, with the ridge penalty chosen by an inner CV on
each training fold (no leakage). Uses `Xf` and `y` from aspect 5.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def make_model():
    # standardize -> ridge with penalty picked by internal (nested) CV
    return make_pipeline(StandardScaler(),
                         RidgeCV(alphas=np.logspace(-3, 3, 13)))

cv = KFold(n_splits=5, shuffle=True, random_state=0)

def cv_pred_r(X, y_):
    """Cross-validated predictions and their Pearson r with the target."""
    y_pred = cross_val_predict(make_model(), X, y_, cv=cv)
    return np.corrcoef(y_, y_pred)[0, 1], y_pred

# In-sample vs out-of-sample, side by side
r_in = np.corrcoef(y, make_model().fit(Xf, y).predict(Xf))[0, 1]
obs_r, y_pred = cv_pred_r(Xf, y)
r2_oos = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

print(f"  n subjects         : {len(y)}")
print(f"In-sample r          : {r_in:.3f}   (optimistic)")
print(f"Out-of-sample r (CV) : {obs_r:.3f}")
print(f"Out-of-sample R^2    : {r2_oos:.3f}   (can be negative)")

# Task analyses

Description of each task, task timing, and conditions is located [here](https://protocols.humanconnectome.org/HCP/3T/task-fMRI-protocol-details.html).

These are the condition names for each task:

```
- MOTOR: cue, lf, lh, rf, rh, t
- WM:
    0bk_body, 0bk_faces, 0bk_nir, 0bk_placed, 0bk_tools,
    2bk_body, 2bk_faces, 2bk_nir, 2bk_placed, 2bk_tools,
    0bk_cor, 0bk_err,
    2bk_cor, 2bk_err,
    all_bk_cor, all_bk_err
- EMOTION: feat, neutral
- GAMBLING: loss, loss_event, win, win_event, neut_event
- LANGUAGE:
    cue,
    math, story
    present_math, present_story,
    question_math, question_story,
    response_math, response_story
- RELATIONAL: error, match, relation
- SOCIAL: mental_resp, mental, other_resp, rnd
```

## Load individual runs for a given task

Load each subject's data for a specific task, separately for each run:

In [ ]:
timeseries_task = []
for subject in subjects:
  timeseries_task.append(load_timeseries(subject, "motor",
                                         dir=os.path.join(HCP_DIR, "hcp_task"),
                                         concat=False))

## Run a simple subtraction analysis

In [ ]:
help(load_evs)

In [ ]:
help(selective_average)

In [ ]:
task = "WM"
conditions = ["0bk_cor", "2bk_cor"]  # Run a subtraction analysis between two conditions

contrast = []
for subject in subjects:

  # Get the average signal in each region for each condition
  evs = [load_evs(subject, task, cond, dir=os.path.join(HCP_DIR, "hcp_task")) for cond in conditions]
  avgs = [selective_average(timeseries_task[subject], ev) for ev in evs]

  # Store the region-wise difference
  contrast.append(avgs[0] - avgs[1])

group_contrast = np.mean(contrast, axis=0)

Concatenate the 0bk & 2bk

In [ ]:
subject = 0

for cond in [
    "0bk_body",
    "0bk_faces",
    "0bk_places",
    "0bk_tools",
    "2bk_body",
    "2bk_faces",
    "2bk_places",
    "2bk_tools"
]:
    try:
        ev = load_evs(subject, "wm", cond,
                      dir=os.path.join(HCP_DIR, "hcp_task"))
        print(cond, "✓")
    except:
        print(cond, "not found")

In [ ]:
#selective_average(timeseries_data, ev) return the average activation
subject = 0

ev = load_evs(
    subject,
    "wm",
    "0bk_body",
    dir=os.path.join(HCP_DIR, "hcp_task")
)

print(ev)

**we create a function to extract TRs since the selective_average(timeseries_data, ev) give the average activation for Run 1 & Run 2**




In [ ]:
def extract_block(run_data, onset, duration, TR=0.72):
    """
    Extract one task block.

    Parameters
    ----------
    run_data : array (360, nTR)
    onset : float (seconds)
    duration : float (seconds)

    Returns
    -------
    array (360, selected_TRs)
    """
    start = int(np.floor(onset / TR))
    end = int(np.ceil((onset + duration) / TR))

    return run_data[:, start:end]

In [ ]:
subject = 0

timeseries_runs = load_timeseries(
    subject,
    "wm",
    dir=os.path.join(HCP_DIR, "hcp_task"),
    concat=False
)

zero_conditions = [
    "0bk_body",
    "0bk_faces",
    "0bk_places",
    "0bk_tools"
]

zero_blocks = []

for cond in zero_conditions:

    evs = load_evs(
        subject,
        "wm",
        cond,
        dir=os.path.join(HCP_DIR, "hcp_task")
    )

    for run in range(2):

        block = extract_block(
            timeseries_runs[run],
            evs[run]["onset"][0],
            evs[run]["duration"][0],
            TR
        )

        zero_blocks.append(block)

In [ ]:
zero_back = np.concatenate(zero_blocks, axis=1)

print(zero_back.shape)

In [ ]:
two_conditions = [
    "2bk_body",
    "2bk_faces",
    "2bk_places",
    "2bk_tools"
]

two_blocks = []

for cond in two_conditions:

    evs = load_evs(
        subject,
        "wm",
        cond,
        dir=os.path.join(HCP_DIR, "hcp_task")
    )

    for run in range(2):

        block = extract_block(
            timeseries_runs[run],
            evs[run]["onset"][0],
            evs[run]["duration"][0],
            TR
        )

        two_blocks.append(block)

two_back = np.concatenate(two_blocks, axis=1)

print(two_back.shape)

All 100 participants

In [ ]:
all_zero_back = []
all_two_back = []

zero_conditions = [
    "0bk_body",
    "0bk_faces",
    "0bk_places",
    "0bk_tools"
]

two_conditions = [
    "2bk_body",
    "2bk_faces",
    "2bk_places",
    "2bk_tools"
]

for subject in map(int, final_subjects):

    timeseries_runs = load_timeseries(
        subject,
        "wm",
        dir=os.path.join(HCP_DIR, "hcp_task"),
        concat=False
    )

    ...

    # -------- 0-back --------
    zero_blocks = []

    for cond in zero_conditions:

        evs = load_evs(
            subject,
            "wm",
            cond,
            dir=os.path.join(HCP_DIR, "hcp_task")
        )

        for run in range(2):

            block = extract_block(
                timeseries_runs[run],
                evs[run]["onset"][0],
                evs[run]["duration"][0],
                TR
            )

            zero_blocks.append(block)

    zero_back = np.concatenate(zero_blocks, axis=1)

    # -------- 2-back --------
    two_blocks = []

    for cond in two_conditions:

        evs = load_evs(
            subject,
            "wm",
            cond,
            dir=os.path.join(HCP_DIR, "hcp_task")
        )

        for run in range(2):

            block = extract_block(
                timeseries_runs[run],
                evs[run]["onset"][0],
                evs[run]["duration"][0],
                TR
            )

            two_blocks.append(block)

    two_back = np.concatenate(two_blocks, axis=1)

    # Store this subject
    all_zero_back.append(zero_back)
    all_two_back.append(two_back)

In [ ]:
print(len(all_zero_back))
print(len(all_two_back))

In [ ]:
print(all_zero_back[0].shape)
print(all_two_back[0].shape)

Plot group-averaged contrast value across regions:

In [ ]:
hemi_contrasts = np.split(group_contrast, 2)

plt.figure()
for i, hemi_contrast in enumerate(hemi_contrasts):
  plt.plot(hemi_contrast, label=f"{HEMIS[i]} hemisphere")

plt.title(f"Contrast of {conditions[0]} - {conditions[1]}")
plt.xlabel("Region")
plt.ylabel('Contrast')
plt.legend()
plt.show()

Plot the regional values on the surface of one hemisphere:

In [ ]:
fsaverage = datasets.fetch_surf_fsaverage()
surf_contrast = group_contrast[atlas["labels_L"]]
plotting.view_surf(fsaverage['infl_left'], surf_contrast, vmax=15)

## Characterize values by functional network

Average the contrast values within parcels belonging to each network and plot:

In [ ]:
# Get unique network labels
network_names = np.unique(region_info["network"])

hemi_networks = np.split(region_info["network"], 2)
hemi_contrasts = np.split(group_contrast, 2)

plt.figure()
# Get and plot mean contrast value per network, by hemisphere
for hemi, hemi_network, hemi_contrast in zip(HEMIS, hemi_networks, hemi_contrasts):
  network_vals = []
  for network in network_names:
    network_vals.append(hemi_contrast[hemi_network == network].mean())
  plt.barh(network_names, network_vals, alpha=.5, label=f"{hemi} hemi")

plt.axvline(0)
plt.xlabel("Amplitude difference")
plt.title(f"Contrast of {conditions[0]} - {conditions[1]}")
plt.legend()
plt.show()

# Task GLM analysis (General Linear Model)

The subtraction analysis above just averages the raw frames belonging to each
condition. A proper **General Linear Model** instead builds a *design matrix*
(condition onsets/durations convolved with a hemodynamic response function,
plus drift terms), then regresses each parcel's BOLD time series against it.
This deconfounds the conditions from one another and yields not only effect
sizes but z / t statistics.

We reuse the notebook's own `load_timeseries(..., concat=False)` and `load_evs`
together with `nilearn.glm`.

In [ ]:
import os
import pandas as pd
from nilearn.glm.first_level import make_first_level_design_matrix, run_glm
from nilearn.glm.contrasts import compute_contrast


def list_ev_conditions(subject, task, dir):
    """Return the EV condition names actually present on disk for a task."""
    task_key = BOLD_NAMES[get_image_ids(task)[0] - 1]
    ev_dir = os.path.join(dir, "subjects", str(subject), "EVs", task_key)
    return sorted(f[:-4] for f in os.listdir(ev_dir) if f.endswith(".txt"))


def build_events(subject, task, conditions, dir):
    """One events DataFrame (onset / duration / trial_type) per run. (unchanged)"""
    ev_by_cond = {c: load_evs(subject, task, c, dir) for c in conditions}
    n_runs = len(next(iter(ev_by_cond.values())))
    events_per_run = []
    for r in range(n_runs):
        rows = []
        for c in conditions:
            ev = ev_by_cond[c][r]
            for onset, dur in zip(np.atleast_1d(ev["onset"]),
                                  np.atleast_1d(ev["duration"])):
                rows.append({"trial_type": c, "onset": onset, "duration": dur})
        events_per_run.append(pd.DataFrame(rows))
    return events_per_run


def averaged_contrast(design_columns, pos, neg):
    """Contrast vector: +1/|pos| on each pos column, -1/|neg| on each neg column.

    Averaging within each side means the result is a clean 'mean(pos) - mean(neg)',
    padded with zeros for drift/constant columns.
    """
    cols = list(design_columns)
    con = np.zeros(len(cols))
    pos = [c for c in pos if c in cols]
    neg = [c for c in neg if c in cols]
    for c in pos:
        con[cols.index(c)] = 1.0 / len(pos)
    for c in neg:
        con[cols.index(c)] = -1.0 / len(neg)
    return con


def fit_subject_glm(subject, task, model_conditions, pos, neg, dir):
    """Fit a per-run GLM over `model_conditions`, return run-averaged (effect, z)."""
    ts_runs = load_timeseries(subject, task, dir, concat=False)
    events_per_run = build_events(subject, task, model_conditions, dir)

    effects, zscores = [], []
    for ts, events in zip(ts_runs, events_per_run):
        frame_times = TR * np.arange(ts.shape[1])
        X = make_first_level_design_matrix(
            frame_times, events,
            hrf_model="spm", drift_model="cosine", high_pass=0.01,
        )
        Y = ts.T
        labels, estimates = run_glm(Y, X.values, noise_model="ols")

        con_val = averaged_contrast(X.columns, pos, neg)
        con = compute_contrast(labels, estimates, con_val, stat_type="t")
        effects.append(np.asarray(con.effect_size()).ravel())
        zscores.append(np.asarray(con.z_score()).ravel())

    return np.mean(effects, axis=0), np.mean(zscores, axis=0)

Fit the GLM across all subjects. We model **all** MOTOR conditions in a
single design (`cue, lf, lh, rf, rh, t`) and read out the same `lf - rf`
contrast used above, so the result is directly comparable to `group_contrast`.

In [ ]:
task = "wm"
task_dir = os.path.join(HCP_DIR, "hcp_task")

# Discover real EV names — inspect this printout to confirm the category names
avail = list_ev_conditions(0, task, task_dir)
print("Available WM conditions:", avail)

# Category regressors only; drop accuracy / no-response bookkeeping EVs
cat = [c for c in avail
       if (c.startswith("0bk_") or c.startswith("2bk_"))
       and not c.endswith(("_cor", "_err", "_nlr", "_nir"))]
pos = [c for c in cat if c.startswith("2bk_")]   # high load
neg = [c for c in cat if c.startswith("0bk_")]   # low load
print("Modeling:", cat)
print("Load contrast:", pos, "\n            minus", neg)

effects, zscores = [], []
for subject in subjects:
    eff, z = fit_subject_glm(subject, task, cat, pos, neg, task_dir)
    effects.append(eff)
    zscores.append(z)

group_load_contrast = np.mean(effects, axis=0)   # the load map
group_load_z = np.mean(zscores, axis=0)

# task = "motor"
# conditions = ["cue", "lf", "lh", "rf", "rh", "t"]  # model every condition
# contrast_expr = "lf - rf"                          # left vs right foot

# task_dir = os.path.join(HCP_DIR, "hcp_task")

# effects, zscores = [], []
# for subject in subjects:
#     eff, z = fit_subject_glm(subject, task, conditions, contrast_expr, task_dir)
#     effects.append(eff)
#     zscores.append(z)

# group_contrast_glm = np.mean(effects, axis=0)   # comparable to group_contrast
# group_z_glm = np.mean(zscores, axis=0)

# print("GLM contrast shape:", group_contrast_glm.shape)

Plot the GLM group contrast across regions (same layout as the subtraction plot):

In [ ]:
%who

In [ ]:
hemi_contrasts = np.split(group_contrast, 2)

HEMIS = ["Left", "Right"]

plt.figure(figsize=(10,4))
for i, hemi_contrast in enumerate(hemi_contrasts):
    plt.plot(hemi_contrast, label=f"{HEMIS[i]} hemisphere")

plt.title("GLM contrast: 2-back vs 0-back")   # or use {contrast}
plt.xlabel("Region")
plt.ylabel("Effect size (β)")
plt.legend()
plt.show()

In [ ]:
hemi_contrasts = np.split(group_contrast, 2)

plt.figure()
for i, hemi_contrast in enumerate(hemi_contrasts):
    plt.plot(hemi_contrast, label=f"{HEMIS[i]} hemisphere")

plt.title(f"GLM contrast of {contrast_expr}")
plt.xlabel("Region")
plt.ylabel("Effect size (beta)")
plt.legend()
plt.show()

Compare the crude subtraction against the GLM effect size, parcel by parcel:

In [ ]:
plt.figure()
plt.scatter(group_contrast, group_contrast_glm, s=8, alpha=.5)
plt.axhline(0, color="k", lw=.5)
plt.axvline(0, color="k", lw=.5)
plt.xlabel("Subtraction (selective_average)")
plt.ylabel("GLM effect size")
plt.title(f"Subtraction vs GLM, {contrast_expr}")
plt.show()

r = np.corrcoef(group_contrast, group_contrast_glm)[0, 1]
print(f"Correlation between the two approaches: r = {r:.3f}")

Project the GLM z-map onto the left hemisphere surface:

In [ ]:
fsaverage = datasets.fetch_surf_fsaverage()
surf_z = group_z_glm[atlas["labels_L"]]
plotting.view_surf(fsaverage['infl_left'], surf_z, vmax=8)

for task need connectivity .


# Generalized Linear Model (behavioral / demographic prediction)

The section above is the *general* linear model of fMRI. A **generalized**
linear model (statsmodels sense) lets you relate brain-derived features to an
outcome with a non-Gaussian link — logistic for binary, Poisson for counts,
etc. Below we predict one synthetic demographic variable from resting-state
features (mean within-network functional connectivity per subject).

This cell depends on `fc` computed in the resting-state section above.

# Behavioral covariates


## Task performance measures

The dataset also includes aggregate behavior for each task run stored in task-specific `.csv` files. It is possible to load and work with these files using `numpy`:

In [ ]:
wm_behavior = np.genfromtxt(os.path.join(HCP_DIR, "hcp", "behavior", "wm.csv"),
                            delimiter=",",
                            names=True,
                            dtype=None,
                            encoding="utf")
print(wm_behavior[:5])
print(wm_behavior.dtype.names)

But, while not formally taught as part of the course, [`pandas`](https://pandas.pydata.org/) offers more powerful tools for tabular data analysis.

## Pseudo-demographics

The NMA-distributed version of the HCP data does not contain any real demographic information. But we have created a synthetic dataset of 25 "demographic" variables based on a model trained on the original dataset to predict demographics from resting-state network organization measures:

In [ ]:
demo = np.load(os.path.join(HCP_DIR, "hcp", "pseudo_demographics.npy"))
demo.shape

### Fit a Generalized Linear Model

Build subject-level features (mean within-network FC), then fit a statsmodels
GLM. Swap `family=` for the outcome type: `Gaussian` (continuous),
`Binomial` (binary/logistic), or `Poisson` (counts).

In [ ]:
import statsmodels.api as sm

# Feature matrix: mean within-network FC per subject -> (n_subjects x n_networks)
networks = np.unique(region_info["network"])
feat = np.zeros((N_SUBJECTS, len(networks)))
for s in range(N_SUBJECTS):
    for j, net in enumerate(networks):
        idx = np.where(region_info["network"] == net)[0]
        feat[s, j] = fc[s][np.ix_(idx, idx)].mean()

# Target: pick a column of the pseudo-demographics
y = demo[:, 0].astype(float)
Xd = sm.add_constant(feat)

# Gaussian (identity link). Use Binomial() for binary, Poisson() for counts.
model = sm.GLM(y, Xd, family=sm.families.Gaussian())
res = model.fit()
print(res.summary())

Predicted vs observed:

In [ ]:
y_hat = res.predict(Xd)

plt.figure()
plt.scatter(y, y_hat, s=10, alpha=.5)
lims = [min(y.min(), y_hat.min()), max(y.max(), y_hat.max())]
plt.plot(lims, lims, "k--", lw=.7)
plt.xlabel("Observed")
plt.ylabel("Predicted")
plt.title("GLM: observed vs predicted")
plt.show()

## Original subject IDs

The dataset also contains a key to map the ordinal subject numbers to the IDs used in the original HCP dataset:

In [ ]:
ids = np.loadtxt(os.path.join(HCP_DIR, "hcp", "orig_ids.txt"))
print(ids[:8])